In [ ]:
import glob
import pandas as pd
import numpy as np

keep = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "HS", "AS", "HST", "AST",
        "AvgCH", "AvgCD", "AvgCA",
        "PSCH", "PSCD", "PSCA"]

files = glob.glob("dataset/E0*.csv")

df = pd.concat(
    [pd.read_csv(f, encoding="latin-1").reindex(columns=keep) for f in files],
    ignore_index=True,
)

df = df.dropna(subset=["HomeTeam"])
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
df = df.sort_values("Date").reset_index(drop=True)

df["HomeTeam"].value_counts()
df.isna().sum()

Date           0
HomeTeam       0
AwayTeam       0
FTHG           0
FTAG           0
FTR            0
HS             0
AS             0
HST            0
AST            0
AvgCH       1520
AvgCD       1520
AvgCA       1520
PSCH         280
PSCD         280
PSCA         280
dtype: int64

In [4]:
missing = df[df["PSCH"].isna()]
print(missing["Date"].dt.year.value_counts().sort_index()) # Only missing from 2026 season since all games haven't been played yet

miss = df[df["PSCH"].isna()]
print(miss["Date"].min(), miss["Date"].max())

print(pd.read_csv("dataset/E0.csv", encoding="latin-1").columns.tolist())
print(df.shape)
print(pd.read_csv("dataset/E0.csv", encoding="latin-1")[["AvgCH","AvgCD","AvgCA"]].isna().sum())

Date
2026    280
Name: count, dtype: int64
2026-01-17 00:00:00 2026-11-23 00:00:00
['ï»¿Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BFDH', 'BFDD', 'BFDA', 'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BFEH', 'BFED', 'BFEA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BFE>2.5', 'BFE<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'BFEAHH', 'BFEAHA', 'B365CH', 'B365CD', 'B365CA', 'BFDCH', 'BFDCD', 'BFDCA', 'BMGMCH', 'BMGMCD', 'BMGMCA', 'BVCH', 'BVCD', 'BVCA', 'BWCH', 'BWCD', 'BWCA', 'CLCH', 'CLCD', 'CLCA', 'LBCH', 'LBCD', 'LBCA', 'PSCH', 'PSCD', 'PSCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'BFECH

In [7]:
# Initial benchmark model, turning odds into probabilities
bench = df.dropna(subset=["PSCH", "PSCD", "PSCA"]).copy()

bench["pH"] = 1 / bench["PSCH"]
bench["pD"] = 1 / bench["PSCD"]
bench["pA"] = 1 / bench["PSCA"]

total = bench["pH"] + bench["pD"] + bench["pA"]
bench["pH"] = bench["pH"] / total
bench["pD"] = bench["pD"] / total
bench["pA"] = bench["pA"] / total

print(bench[["pH", "pD", "pA"]].sum(axis=1).head())

p_actual = np.where(bench["FTR"] == "H", bench["pH"],
           np.where(bench["FTR"] == "D", bench["pD"], bench["pA"]))

print("Log loss:", -np.log(p_actual).mean())

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64
Log loss: 0.9552114612835595


In [22]:
# Creating the train, test datasets

train = bench[bench['Date'] <= '2024-06-01']
test = bench[bench['Date'] > '2024-06-01']


freq = train["FTR"].value_counts(normalize=True)
print(freq)

base_p = test["FTR"].map(freq)
print("Baseline log loss:", -np.log(base_p).mean())

book_p = np.where(test["FTR"] == "H", test["pH"],
         np.where(test["FTR"] == "D", test["pD"], test["pA"]))
print("Bookmaker log loss:", -np.log(book_p).mean())


FTR
H    0.448830
A    0.321053
D    0.230117
Name: proportion, dtype: float64
Baseline log loss: 1.077720651737262
Bookmaker log loss: 0.9739051929119461


My model log loss: 1.052472150521061
